### Weight Calculations

In [8]:

import numpy as np
from scipy.optimize import linprog

# ==================================================
# 1. INPUT DATA - FINAL MANAGER COMPARISONS
# ==================================================

best_to_others = np.array([1, 7, 3, 4, 5], dtype=float)
others_to_worst = np.array([7, 1, 5, 3, 2], dtype=float)

B = 0  # Best criterion: C1 (OTIF)
W = 1  # Worst criterion: C2 (Holding Cost)
n = 5


# ==================================================
# 2. BUILD LINEAR CONSTRAINTS
# ==================================================

def build_constraints(xi):

    rows = []

    # Best-to-others constraints
    for j in range(n):

        if j != B:

            r = np.zeros(n)
            r[B] = 1
            r[j] = -(best_to_others[j] + xi)
            rows.append(r)

            r = np.zeros(n)
            r[B] = -1
            r[j] = best_to_others[j] - xi
            rows.append(r)

    # Others-to-worst constraints
    for j in range(n):

        if j != W:

            r = np.zeros(n)
            r[j] = 1
            r[W] = -(others_to_worst[j] + xi)
            rows.append(r)

            r = np.zeros(n)
            r[j] = -1
            r[W] = others_to_worst[j] - xi
            rows.append(r)

    return np.array(rows)


# ==================================================
# 3. FEASIBILITY FUNCTION
# ==================================================

def feasible(xi, objective=None):

    A_ub = build_constraints(xi)

    c = np.zeros(n) if objective is None else objective

    return linprog(
        c,
        A_ub=A_ub,
        b_ub=np.zeros(len(A_ub)),
        A_eq=[np.ones(n)],
        b_eq=[1.0],
        bounds=[(1e-9, None)] * n,
        method="highs"
    )


# ==================================================
# 4. FIND OPTIMAL MAXIMUM DEVIATION (XI*)
# ==================================================

lo = 0.0
hi = 20.0

if not feasible(hi).success:
    raise RuntimeError("Initial upper bound is not feasible.")

for _ in range(100):

    mid = (lo + hi) / 2

    result = feasible(mid)

    if result.success:
        hi = mid
    else:
        lo = mid

xi_star = hi

print("OPTIMAL BWM DEVIATION")
print("---------------------")
print(f"Xi* = {xi_star:.6f}")


# ==================================================
# 5. CALCULATE MINIMUM AND MAXIMUM WEIGHT RANGES
# ==================================================

# Small numerical tolerance for LP feasibility
xi_range = xi_star + 1e-5

lower = np.zeros(n)
upper = np.zeros(n)

for j in range(n):

    e = np.zeros(n)
    e[j] = 1

    # Minimise the weight of criterion j
    result_min = feasible(xi_range, objective=e)

    # Maximise the weight of criterion j
    result_max = feasible(xi_range, objective=-e)

    if not result_min.success:
        raise RuntimeError(
            f"Minimum weight calculation failed for C{j+1}: "
            f"{result_min.message}"
        )

    if not result_max.success:
        raise RuntimeError(
            f"Maximum weight calculation failed for C{j+1}: "
            f"{result_max.message}"
        )

    lower[j] = result_min.x[j]
    upper[j] = result_max.x[j]


# ==================================================
# 6. CALCULATE CENTRAL WEIGHTS
# ==================================================

# Midpoint of each individual weight interval
centre = (lower + upper) / 2

# Normalise so that all weights sum to 1
weights = centre / centre.sum()


# ==================================================
# 7. DISPLAY WEIGHT RANGES
# ==================================================

criteria = [
    "C1: OTIF",
    "C2: Holding Cost",
    "C3: Obsolescence",
    "C4: Implementation Cost",
    "C5: Implementation Time"
]

print("\nWEIGHT RANGES")
print("-" * 65)

for j in range(n):

    print(
        f"{criteria[j]:30s} "
        f"Min = {lower[j]:.4f} | "
        f"Max = {upper[j]:.4f}"
    )


# ==================================================
# 8. DISPLAY FINAL CENTRALISED WEIGHTS
# ==================================================

print("\nFINAL CENTRALISED BWM WEIGHTS")
print("-" * 65)

for j in range(n):

    print(
        f"{criteria[j]:30s} "
        f"Weight = {weights[j]:.4f} | "
        f"Percentage = {weights[j]*100:.2f}%"
    )

print("-" * 65)
print(f"Sum of weights = {weights.sum():.6f}")


# ==================================================
# 9. CHECK FINAL WEIGHT RATIOS AND DEVIATIONS
# ==================================================

print("\nBWM DEVIATION CHECK")
print("-" * 65)

for j in range(n):

    if j != B:

        ratio = weights[B] / weights[j]
        deviation = abs(ratio - best_to_others[j])

        print(
            f"Best-to-C{j+1}: "
            f"Ratio = {ratio:.4f}, "
            f"Deviation = {deviation:.4f}"
        )

for j in range(n):

    if j != W:

        ratio = weights[j] / weights[W]
        deviation = abs(ratio - others_to_worst[j])

        print(
            f"C{j+1}-to-Worst: "
            f"Ratio = {ratio:.4f}, "
            f"Deviation = {deviation:.4f}"
        )


OPTIMAL BWM DEVIATION
---------------------
Xi* = 1.000000

WEIGHT RANGES
-----------------------------------------------------------------
C1: OTIF                       Min = 0.4528 | Max = 0.4898
C2: Holding Cost               Min = 0.0566 | Max = 0.0612
C3: Obsolescence               Min = 0.2264 | Max = 0.2449
C4: Implementation Cost        Min = 0.1176 | Max = 0.1569
C5: Implementation Time        Min = 0.0784 | Max = 0.1176

FINAL CENTRALISED BWM WEIGHTS
-----------------------------------------------------------------
C1: OTIF                       Weight = 0.4708 | Percentage = 47.08%
C2: Holding Cost               Weight = 0.0588 | Percentage = 5.88%
C3: Obsolescence               Weight = 0.2354 | Percentage = 23.54%
C4: Implementation Cost        Weight = 0.1371 | Percentage = 13.71%
C5: Implementation Time        Weight = 0.0979 | Percentage = 9.79%
-----------------------------------------------------------------
Sum of weights = 1.000000

BWM DEVIATION CHECK
------------

### Local Concordance Calculation

In [1]:

import numpy as np
import pandas as pd

# Alternatives
alternatives = ["A0", "A1", "A2", "A3", "A4"]

# Performance matrix
# Columns: C1, C2, C3, C4, C5

performance = np.array([
    [0.78, 22_000_000, 440_000,      0, 0],
    [0.88, 21_500_000, 370_000, 30_000, 3],
    [0.83, 18_000_000, 320_000, 80_000, 5],
    [0.89, 20_000_000, 340_000, 40_000, 2],
    [0.87, 24_000_000, 500_000, 50_000, 5]
])

criteria = [
    "C1 - OTIF",
    "C2 - Holding cost",
    "C3 - Obsolescence",
    "C4 - Implementation cost",
    "C5 - Implementation time"
]

# Direction: 1 = benefit, -1 = cost
direction = np.array([1, -1, -1, -1, -1])

# Indifference thresholds
q = np.array([
    0.03,
    2_000_000,
    60_000,
    20_000,
    4
])

# Preference thresholds
p = np.array([
    0.10,
    5_000_000,
    220_000,
    50_000,
    10
])

# Calculate local concordance
local_matrices = {}

for j in range(5):

    # Difference in favour of row alternative A
    # compared with column alternative B
    difference = direction[j] * (
        performance[:, j, None]
        - performance[None, :, j]
    )

    concordance = np.where(
        difference >= -q[j],
        1,
        np.where(
            difference <= -p[j],
            0,
            (difference + p[j]) / (p[j] - q[j])
        )
    )

    # An alternative is not compared with itself
    np.fill_diagonal(concordance, np.nan)

    local_matrices[criteria[j]] = pd.DataFrame(
        concordance,
        index=alternatives,
        columns=alternatives
    )

# Display all five matrices
for criterion, matrix in local_matrices.items():

    print("\n", criterion)
    print(matrix.round(4))



 C1 - OTIF
     A0      A1      A2      A3      A4
A0  NaN  0.0000  0.7143  0.0000  0.1429
A1  1.0     NaN  1.0000  1.0000  1.0000
A2  1.0  0.7143     NaN  0.5714  0.8571
A3  1.0  1.0000  1.0000     NaN  1.0000
A4  1.0  1.0000  1.0000  1.0000     NaN

 C2 - Holding cost
     A0      A1      A2      A3   A4
A0  NaN  1.0000  0.3333  1.0000  1.0
A1  1.0     NaN  0.5000  1.0000  1.0
A2  1.0  1.0000     NaN  1.0000  1.0
A3  1.0  1.0000  1.0000     NaN  1.0
A4  1.0  0.8333  0.0000  0.3333  NaN

 C3 - Obsolescence
     A0      A1     A2     A3   A4
A0  NaN  0.9375  0.625  0.750  1.0
A1  1.0     NaN  1.000  1.000  1.0
A2  1.0  1.0000    NaN  1.000  1.0
A3  1.0  1.0000  1.000    NaN  1.0
A4  1.0  0.5625  0.250  0.375  NaN

 C4 - Implementation cost
        A0   A1   A2      A3      A4
A0     NaN  1.0  1.0  1.0000  1.0000
A1  0.6667  NaN  1.0  1.0000  1.0000
A2  0.0000  0.0  NaN  0.3333  0.6667
A3  0.3333  1.0  1.0     NaN  1.0000
A4  0.0000  1.0  1.0  1.0000     NaN

 C5 - Implementation time


### Discordance, Global Concordance, Credibility Calculation

In [9]:
import numpy as np
import pandas as pd

# Alternatives and criteria
alternatives = ["A0", "A1", "A2", "A3", "A4"]

criteria = [
    "C1 - OTIF",
    "C2 - Holding cost",
    "C3 - Obsolescence",
    "C4 - Implementation cost",
    "C5 - Implementation time"
]

# Performance data: rows = alternatives, columns = criteria
performance = np.array([
    [0.78, 22_000_000, 440_000,      0, 0],
    [0.88, 21_500_000, 370_000, 30_000, 3],
    [0.83, 18_000_000, 320_000, 80_000, 5],
    [0.89, 20_000_000, 340_000, 40_000, 2],
    [0.87, 24_000_000, 500_000, 50_000, 5]
])

# BWM weights selected in the previous step
weights = np.array([0.4708,
0.0588,
0.2354,
0.1371,
0.0979
])

# Benefit = 1; Cost = -1
direction = np.array([1, -1, -1, -1, -1])

# Thresholds
q = np.array([0.03, 2_000_000, 60_000, 20_000, 4])
p = np.array([0.10, 5_000_000, 220_000, 50_000, 10])

# Veto thresholds: None means no veto
v = [0.20, 8_000_000, 440_000, None, None]

n_alt = len(alternatives)
n_criteria = len(criteria)

# Difference in favour of row alternative A
# compared with column alternative B
difference = np.stack([
    direction[j] * (
        performance[:, j, None] - performance[None, :, j]
    )
    for j in range(n_criteria)
], axis=2)

# ----------------------------------
# 1. Calculate local concordance
# ----------------------------------

local_concordance = np.zeros(
    (n_alt, n_alt, n_criteria)
)

for j in range(n_criteria):
    d = difference[:, :, j]

    local_concordance[:, :, j] = np.where(
        d >= -q[j],
        1,
        np.where(
            d <= -p[j],
            0,
            (d + p[j]) / (p[j] - q[j])
        )
    )

# ----------------------------------
# 2. Calculate global concordance
# ----------------------------------

global_concordance = np.sum(
    local_concordance * weights[None, None, :],
    axis=2
)

np.fill_diagonal(global_concordance, np.nan)

# ----------------------------------
# 3. Calculate local discordance
# ----------------------------------

discordance = np.zeros(
    (n_alt, n_alt, n_criteria)
)

for j in range(n_criteria):

    # Skip criteria without veto thresholds
    if v[j] is None:
        continue

    d = difference[:, :, j]

    discordance[:, :, j] = np.where(
        d <= -v[j],
        1,
        np.where(
            d >= -p[j],
            0,
            (d + p[j]) / (p[j] - v[j])
        )
    )

    np.fill_diagonal(discordance[:, :, j], np.nan)

# ----------------------------------
# 4. Calculate credibility
# ----------------------------------

credibility = np.full(
    (n_alt, n_alt), np.nan
)

adjustment_factors = np.ones(
    (n_alt, n_alt, n_criteria)
)

for a in range(n_alt):
    for b in range(n_alt):

        if a == b:
            continue

        C = global_concordance[a, b]

        for j in range(n_criteria):

            # No veto criterion: adjustment = 1
            if v[j] is None:
                continue

            D = discordance[a, b, j]

            if D > C:
                adjustment_factors[a, b, j] = (
                    (1 - D) / (1 - C)
                )

        credibility[a, b] = (
            C * np.prod(adjustment_factors[a, b, :])
        )

# ----------------------------------
# 5. Display results
# ----------------------------------

for j in range(n_criteria):

    if v[j] is None:
        print(f"\n{criteria[j]}: No veto threshold")
        continue

    print(f"\nDISCORDANCE - {criteria[j]}")

    print(pd.DataFrame(
        discordance[:, :, j],
        index=alternatives,
        columns=alternatives
    ).round(4))

print("\nGLOBAL CONCORDANCE")
print(pd.DataFrame(
    global_concordance,
    index=alternatives,
    columns=alternatives
).round(4))

print("\nCREDIBILITY MATRIX")
print(pd.DataFrame(
    credibility,
    index=alternatives,
    columns=alternatives
).round(4))

print("\nCREDIBILITY EQUALS GLOBAL CONCORDANCE:",
      np.allclose(
          credibility[~np.isnan(credibility)],
          global_concordance[~np.isnan(global_concordance)]
      ))



DISCORDANCE - C1 - OTIF
     A0   A1   A2   A3   A4
A0  NaN  0.0  0.0  0.1  0.0
A1  0.0  NaN  0.0  0.0  0.0
A2  0.0  0.0  NaN  0.0  0.0
A3  0.0  0.0  0.0  NaN  0.0
A4  0.0  0.0  0.0  0.0  NaN

DISCORDANCE - C2 - Holding cost
     A0   A1      A2   A3   A4
A0  NaN  0.0  0.0000  0.0  0.0
A1  0.0  NaN  0.0000  0.0  0.0
A2  0.0  0.0     NaN  0.0  0.0
A3  0.0  0.0  0.0000  NaN  0.0
A4  0.0  0.0  0.3333  0.0  NaN

DISCORDANCE - C3 - Obsolescence
     A0   A1   A2   A3   A4
A0  NaN  0.0  0.0  0.0  0.0
A1  0.0  NaN  0.0  0.0  0.0
A2  0.0  0.0  NaN  0.0  0.0
A3  0.0  0.0  0.0  NaN  0.0
A4  0.0  0.0  0.0  0.0  NaN

C4 - Implementation cost: No veto threshold

C5 - Implementation time: No veto threshold

GLOBAL CONCORDANCE
        A0      A1      A2      A3      A4
A0     NaN  0.5145  0.7380  0.4703  0.5965
A1  0.9543     NaN  0.9706  1.0000  1.0000
A2  0.8466  0.7284     NaN  0.7068  0.8870
A3  0.9086  1.0000  1.0000     NaN  1.0000
A4  0.8466  0.8872  0.7646  0.8137     NaN

CREDIBILITY MATRIX

### ELECTRE III Ranking

In [14]:

import numpy as np
import pandas as pd

alternatives = ["A0", "A1", "A2", "A3", "A4"]

# Credibility matrix
credibility = np.array([
    [np.nan, 0.5145, 0.7380, 0.4703, 0.5965],
    [0.9543, np.nan, 0.9706, 1.0000, 1.0000],
    [0.8466, 0.7284, np.nan, 0.7068, 0.8870],
    [0.9086, 1.0000, 1.0000, np.nan, 1.0000],
    [0.8466, 0.8872, 0.7646, 0.8137, np.nan]
])

# Lambda threshold
lam = 0.75
'''Can change this value to see the sensitivity of the outranking results.'''


# Convert credibility matrix into binary outranking matrix
outranking = (credibility >= lam).astype(float)
np.fill_diagonal(outranking, np.nan)

# Replace diagonal NaN with zero for counting
binary = np.nan_to_num(outranking, nan=0.0)

# Calculate outgoing and incoming counts
outgoing = binary.sum(axis=1)
incoming = binary.sum(axis=0)

# Net outranking flow
net_flow = outgoing - incoming

results = pd.DataFrame({
    "Alternative": alternatives,
    "Outgoing count": outgoing.astype(int),
    "Incoming count": incoming.astype(int),
    "Net flow": net_flow.astype(int)
})

results["Rank"] = (
    results["Net flow"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print("BINARY OUTRANKING MATRIX")
print(pd.DataFrame(
    binary,
    index=alternatives,
    columns=alternatives
).astype(int))

print("\nFINAL RANKING")
print(results.sort_values("Rank").to_string(index=False))


BINARY OUTRANKING MATRIX
    A0  A1  A2  A3  A4
A0   0   0   0   0   0
A1   1   0   1   1   1
A2   1   0   0   0   1
A3   1   1   1   0   1
A4   1   1   1   1   0

FINAL RANKING
Alternative  Outgoing count  Incoming count  Net flow  Rank
         A1               4               2         2     1
         A3               4               2         2     1
         A4               4               3         1     3
         A2               2               3        -1     4
         A0               0               4        -4     5


### Sensitivity analysis for weights. . . 

In [15]:

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. INPUT DATA
# --------------------------------------------------

alternatives = ["A0", "A1", "A2", "A3", "A4"]

criteria = [
    "OTIF",
    "Holding Cost",
    "Obsolescence",
    "Implementation Cost",
    "Implementation Time"
]

# Performance matrix
# Rows = alternatives, columns = criteria
performance = np.array([
    [0.78, 22_000_000, 440_000,      0, 0],
    [0.88, 21_500_000, 370_000, 30_000, 3],
    [0.83, 18_000_000, 320_000, 80_000, 5],
    [0.89, 20_000_000, 340_000, 40_000, 2],
    [0.87, 24_000_000, 500_000, 50_000, 5]
], dtype=float)

# Updated centralised BWM weights
weights = np.array([
    0.4708,  # C1: OTIF
    0.0588,  # C2: Holding Cost
    0.2354,  # C3: Obsolescence
    0.1371,  # C4: Implementation Cost
    0.0979   # C5: Implementation Time
])

# ELECTRE thresholds
q = np.array([
    0.03,
    2_000_000,
    60_000,
    20_000,
    4
], dtype=float)

p = np.array([
    0.10,
    5_000_000,
    220_000,
    50_000,
    10
], dtype=float)

# No veto threshold for C4 and C5
v = np.array([
    0.20,
    8_000_000,
    440_000,
    np.inf,
    np.inf
], dtype=float)

# Direction:
# +1 = benefit criterion (higher is better)
# -1 = cost criterion (lower is better)
direction = np.array([1, -1, -1, -1, -1])

lambda_cut = 0.70


# --------------------------------------------------
# 2. ELECTRE CALCULATION FUNCTION
# --------------------------------------------------

def calculate_electre(current_weights):

    n = len(alternatives)
    m = len(criteria)

    concordance_matrix = np.zeros((n, n))
    credibility_matrix = np.zeros((n, n))

    for a in range(n):
        for b in range(n):

            if a == b:
                continue

            # Positive difference means alternative a
            # performs better than alternative b.
            difference = direction * (
                performance[a] - performance[b]
            )

            # --------------------------------------
            # Local concordance
            # --------------------------------------

            local_concordance = np.zeros(m)

            for j in range(m):

                if difference[j] >= -q[j]:
                    local_concordance[j] = 1

                elif difference[j] <= -p[j]:
                    local_concordance[j] = 0

                else:
                    local_concordance[j] = (
                        difference[j] + p[j]
                    ) / (p[j] - q[j])

            # Global concordance
            C = np.dot(current_weights, local_concordance)

            concordance_matrix[a, b] = C

            # --------------------------------------
            # Local discordance
            # --------------------------------------

            local_discordance = np.zeros(m)

            for j in range(m):

                # No veto threshold
                if np.isinf(v[j]):
                    local_discordance[j] = 0

                elif difference[j] <= -v[j]:
                    local_discordance[j] = 1

                elif difference[j] >= -p[j]:
                    local_discordance[j] = 0

                else:
                    local_discordance[j] = (
                        difference[j] + p[j]
                    ) / (p[j] - v[j])

            # --------------------------------------
            # Credibility calculation
            # --------------------------------------

            credibility = C

            for j in range(m):

                d = local_discordance[j]

                if d > C:

                    if C < 1:
                        credibility *= (1 - d) / (1 - C)

            credibility_matrix[a, b] = credibility

    # ----------------------------------------------
    # Lambda-cut outranking matrix
    # ----------------------------------------------

    outranking_matrix = (
        credibility_matrix >= lambda_cut
    ).astype(int)

    np.fill_diagonal(outranking_matrix, 0)

    # Qualification score:
    # Number of alternatives outranked
    # minus number of alternatives that outrank it.

    outgoing = outranking_matrix.sum(axis=1)
    incoming = outranking_matrix.sum(axis=0)

    scores = outgoing - incoming

    return concordance_matrix, credibility_matrix, outranking_matrix, scores


# --------------------------------------------------
# 3. CREATE WEIGHT SENSITIVITY SCENARIOS
# --------------------------------------------------

def change_weight(original_weights, criterion_index, factor):

    new_weights = original_weights.copy()

    old_weight = new_weights[criterion_index]

    # Increase or decrease the selected weight
    new_weights[criterion_index] = old_weight * factor

    # Proportionally rescale all remaining weights
    other_indices = [
        i for i in range(len(new_weights))
        if i != criterion_index
    ]

    remaining_scale = (
        (1 - new_weights[criterion_index])
        / (1 - old_weight)
    )

    new_weights[other_indices] *= remaining_scale

    return new_weights


scenarios = {
    "Original weights": weights,

    "C1 reduced by 20%":
        change_weight(weights, 0, 0.80),

    "C1 increased by 20%":
        change_weight(weights, 0, 1.20),

    "C3 reduced by 20%":
        change_weight(weights, 2, 0.80),

    "C3 increased by 20%":
        change_weight(weights, 2, 1.20)
}


# --------------------------------------------------
# 4. RUN SENSITIVITY ANALYSIS
# --------------------------------------------------

results = []

for scenario_name, scenario_weights in scenarios.items():

    C, credibility, outranking, scores = (
        calculate_electre(scenario_weights)
    )

    result = {
        "Scenario": scenario_name,
        "C1 weight (%)": scenario_weights[0] * 100,
        "C3 weight (%)": scenario_weights[2] * 100,
        "A0": scores[0],
        "A1": scores[1],
        "A2": scores[2],
        "A3": scores[3],
        "A4": scores[4]
    }

    results.append(result)

    print("\n" + "=" * 60)
    print(scenario_name)
    print("=" * 60)

    print("\nUpdated weights:")

    for i, criterion in enumerate(criteria):
        print(
            f"{criterion:25s}: "
            f"{scenario_weights[i]:.6f} "
            f"({scenario_weights[i] * 100:.2f}%)"
        )

    print("\nQualification scores:")

    for i, alternative in enumerate(alternatives):
        print(f"{alternative}: {scores[i]}")

    print("\nLambda-cut outranking matrix:")

    print(pd.DataFrame(
        outranking,
        index=alternatives,
        columns=alternatives
    ))

    # Sort alternatives by qualification score
    ranking = sorted(
        zip(alternatives, scores),
        key=lambda x: x[1],
        reverse=True
    )

    print("\nRanking:")

    for alternative, score in ranking:
        print(f"{alternative}: {score}")


# --------------------------------------------------
# 5. SUMMARY TABLE FOR THE REPORT
# --------------------------------------------------

summary = pd.DataFrame(results)

print("\n\nFINAL WEIGHT SENSITIVITY SUMMARY")
print("=" * 80)

print(summary.to_string(
    index=False,
    formatters={
        "C1 weight (%)": "{:.2f}".format,
        "C3 weight (%)": "{:.2f}".format
    }
))



Original weights

Updated weights:
OTIF                     : 0.470800 (47.08%)
Holding Cost             : 0.058800 (5.88%)
Obsolescence             : 0.235400 (23.54%)
Implementation Cost      : 0.137100 (13.71%)
Implementation Time      : 0.097900 (9.79%)

Qualification scores:
A0: -3
A1: 1
A2: 0
A3: 1
A4: 1

Lambda-cut outranking matrix:
    A0  A1  A2  A3  A4
A0   0   0   1   0   0
A1   1   0   1   1   1
A2   1   1   0   1   1
A3   1   1   1   0   1
A4   1   1   1   1   0

Ranking:
A1: 1
A3: 1
A4: 1
A2: 0
A0: -3

C1 reduced by 20%

Updated weights:
OTIF                     : 0.376640 (37.66%)
Holding Cost             : 0.069262 (6.93%)
Obsolescence             : 0.277284 (27.73%)
Implementation Cost      : 0.161494 (16.15%)
Implementation Time      : 0.115319 (11.53%)

Qualification scores:
A0: -3
A1: 1
A2: 0
A3: 1
A4: 1

Lambda-cut outranking matrix:
    A0  A1  A2  A3  A4
A0   0   0   1   0   0
A1   1   0   1   1   1
A2   1   1   0   1   1
A3   1   1   1   0   1
A4   1   1   1  